# 03 — Streaming Transformations (Bronze → Silver)

Reads from the Bronze table as a stream, applies business logic
transformations, and writes to a Silver Delta table.

In [ ]:
from databricks.connect import DatabricksSession
from pyspark.sql import functions as F
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "streaming_demo")

BRONZE_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.slot_events_bronze"
SILVER_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.slot_events_silver"
CHECKPOINT_PATH = f"/Volumes/{UC_CATALOG}/{UC_SCHEMA}/casino_raw_events/_checkpoints/silver"

## Read Bronze as a stream and apply transformations

In [ ]:
bronze_stream = spark.readStream.table(BRONZE_TABLE)

silver_stream = (
    bronze_stream
    # Cast string columns to proper types (Auto Loader infers JSON values as strings)
    .withColumn("bet_amount", F.col("bet_amount").cast("double"))
    .withColumn("win_amount", F.col("win_amount").cast("double"))
    # Parse the event timestamp
    .withColumn("event_ts", F.to_timestamp("event_timestamp"))
    # Calculate net outcome (positive = player won, negative = house won)
    .withColumn("net_outcome", F.round(F.col("win_amount") - F.col("bet_amount"), 2))
    # Categorize the bet size for easy segmentation
    .withColumn(
        "bet_tier",
        F.when(F.col("bet_amount") < 10, "Low")
        .when(F.col("bet_amount") < 100, "Medium")
        .when(F.col("bet_amount") < 500, "High")
        .otherwise("Whale")
    )
    # Flag whether the play was a win or loss
    .withColumn("is_win", F.col("win_amount") > 0)
    # Extract hour for time-of-day analysis
    .withColumn("event_hour", F.hour("event_ts"))
    # Drop the original string timestamp and rescued data column
    .drop("event_timestamp", "_rescued_data")
)

print("Silver schema:")
silver_stream.printSchema()

## Write to Silver Delta table

In [ ]:
query = (
    silver_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .toTable(SILVER_TABLE)
)

query.awaitTermination()
print(f"Silver table ready: {SILVER_TABLE}")

## Preview Silver data

In [ ]:
silver_df = spark.table(SILVER_TABLE)
print(f"Silver table row count: {silver_df.count()}")
silver_df.show(10, truncate=False)